---

<div align="center">
  <img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/python/python-original.svg" width="80"/>
</div>

<h1 align="center">GenAI & Advanced Nets: O Mecanismo de Atenção </h1>

<h3 align="center">PhD. Julles Mitoura</h3>

<div align="center">
  <img src="https://img.shields.io/badge/Python-3776AB?style=for-the-badge&logo=python&logoColor=white"/>
  <img src="https://img.shields.io/badge/Jupyter-F37626?style=for-the-badge&logo=jupyter&logoColor=white"/>
</div>

---

In [ ]:
# Obs: se você não estiver utilizando um ambiente virtual, instale as bibliotecas conforme se apresenta abaixo
#%pip install -q -r requirements.txt

# pip é o gerenciador de pacotes do Python. Pense nele como o instalador oficial de libs Python.
# no notebook, usar %pip ... é ideal porque instala no mesmo ambiente do kernel em uso.

# -q: quiet
# -r: requirement file, indica ao pip para instalar os pacotes listados no arquivo requirements.txt


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


---

<div align="center">

## <span style="color:#1E90FF;">Anatomia da Arquitetura Transformer do Zero</span>

</div>

Na etapa anterior, desenvolvemos a arquitetura Transformer com PyTorch, aproveitando recursos prontos da biblioteca. Agora, o objetivo é implementar em Python, do zero, os principais módulos dessa arquitetura.

No modelo anterior, utilizamos o `nn.MultiheadAttention`. Embora pareça uma chamada simples, esse módulo do `torch.nn` encapsula várias operações internas. A Figura 1 ajuda a visualizar essa diferença entre a implementação pronta e a construção manual do mecanismo de atenção.

<div style="text-align: center;">
<p><em>Figura 1: Single-Head-Attention e Multi-Head Attention.</em></p>
  <img src="imgs/img03.png" alt="Arquitetura de Transformadores" width="500"/>
</div>

O objetivo agora será desenvolver em Python uma `Single-Head Attention`. Com isso, teremos mais visão sobre os detalhes desta arquitetura e poderemos seguir com o desenvolvimento.

---

Objetivo: Criar um modelo capaz de prever sequencias de comprimento de 1 a 10 tokens.

O modelo transformer será constituido por:
1. Camada de embedding
2. Mecanismo de atenção
3. Camandas Encoder e Decoder
4. Camadas Linear e Softmax

### 1. Hiperparâmetros Iniciais

In [2]:
# imports fundamentis
import numpy as np

# dimensão do modelo
dim_model = 64
# comprimento da sequencia que será predita pelo modelo
seq_length = 10
# tamanho do vocabulario
vocab_size = 100   

#### `dim_model` (dimensão do modelo)

Define o tamanho dos vetores internos usados para representar cada token (embeddings e estados ocultos).

- **Se aumentar `dim_model`**:
  - melhora a capacidade de representação;
  - aumenta custo computacional e uso de memória;
  - pode elevar risco de overfitting em bases pequenas.
- **Se diminuir `dim_model`**:
  - reduz custo e acelera treino/inferência;
  - pode limitar a capacidade do modelo (underfitting).

#### `seq_length` (comprimento da sequência)

Define quantos tokens o modelo observa por amostra (janela de contexto).

- **Se aumentar `seq_length`**:
  - permite capturar dependências mais longas no contexto;
  - aumenta bastante o custo da atenção (aprox. quadrático em relação ao tamanho da sequência);
  - exige mais memória.
- **Se diminuir `seq_length`**:
  - torna o treino mais rápido e barato;
  - reduz o contexto disponível para predição.

#### `vocab_size` (tamanho do vocabulário)

Define quantos tokens distintos existem no espaço de entrada/saída do modelo.

- **Se aumentar `vocab_size`**:
  - melhora cobertura de tokens possíveis;
  - aumenta o tamanho da camada de saída e o custo do treinamento.
- **Se diminuir `vocab_size`**:
  - reduz custo computacional;
  - pode perder informação e aumentar ambiguidades (mais colisões de token).

### 2. Camada de Embeddings

A função de embedding tem por objetivo converter entradas sequenciais em ventores densos de tamanho fixo. Esses vetores são conhecidos como embeddings e são parte fundamental da arquiteture que estamos desenvolvendo.

In [3]:
# função para criar embeddings de uma sequência de tokens
# entrada: lista/array de IDs de token
# saída: matriz (seq_length, dim_model)

def embeddings(input_ids, vocab_size, dim_model):
    # crie uma matriz de embedding onde cada linha representa um token do vocabulário
    # a matriz é iniciada com valores aleatórios normalmente distribuidos
    embed = np.random.randn(vocab_size, dim_model)

    # para cada indice do token input, seleciona-se o embedding correspondente da matriz
    # retorna um array de embeddings correspondentes à sequencia de entrada
    return np.array([embed[i] for i in input_ids])


# exemplo de uso
input_exemplo = [3, 10, 7, 25, 1]
saida_embeddings = embeddings(input_exemplo, vocab_size=vocab_size, dim_model=dim_model)

print("Tokens de entrada:", input_exemplo)
print("Shape da saída:", saida_embeddings.shape)  # (5, 64)
print("Primeiro vetor de embedding (token 3):")
print(saida_embeddings[0])

Tokens de entrada: [3, 10, 7, 25, 1]
Shape da saída: (5, 64)
Primeiro vetor de embedding (token 3):
[-0.67384021 -0.94922336 -0.08993943  1.08995635 -0.89071065 -0.41602603
  1.31768045  1.75145002 -1.65086678  1.2239971  -0.86012389  0.30415134
  0.06984079 -1.10543643  0.60560744  0.06590495  0.08858826  0.52985527
 -0.4942887   0.34106085  0.87330406  0.42885505 -1.77343145 -1.11400526
  1.14553738  1.68372263 -0.90260258 -0.45595604  0.61631991 -0.53312837
  0.11833232 -0.44478884  1.77140203 -0.59387603  0.44735332 -0.10137749
  0.86309026  0.84409455 -0.42900042 -0.29292746  0.60268372 -0.46909077
  0.56018793  0.24352392  1.04443848  1.39405751  0.49824414 -1.41932367
  0.48168761 -0.43073013 -0.77378669 -1.86221446 -1.08762619 -0.96390432
  0.95884792  0.38657589  0.33950314  1.09551241  0.17986057  0.1207579
  1.66333098  0.16212558 -1.51409848 -1.3336321 ]


### 3. Camada de Atenção

Buscaremos construir o mecanismo de atenção como é apresentado na Figura 2.

<div style="text-align: center;">
<p><em>Figura 2: Single-Head-Attention.</em></p>
  <img src="imgs/img04.png" alt="Arquitetura de Transformadores" width="300"/>
</div>

No Transformer, os vetores `Q`, `K` e `V` são projeções lineares usadas para calcular atenção, mas a origem deles depende da subcamada:

- **Self-attention no encoder:** `Q`, `K` e `V` vêm da mesma entrada (`X`);
- **Self-attention no decoder:** `Q`, `K` e `V` também vêm da mesma entrada do decoder (com máscara causal);
- **Cross-attention no decoder:** `Q` vem da saída da subcamada anterior do decoder, enquanto `K` e `V` vêm da saída do encoder.

Antes das projeções, $X$ representa a matriz de entrada da atenção (embeddings dos tokens, geralmente já combinados com informação posicional). Se a sequência possui $n$ tokens e dimensão $d_{\text{model}}$, então:

$$
X \in \mathbb{R}^{n \times d_{\text{model}}}
$$

Formalmente:

$$
\begin{aligned}
Q &= XW_Q \\
K &= XW_K \\
V &= XW_V
\end{aligned}
$$

A atenção é calculada por:

$$
\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

em que os escores entre `Q` e `K` definem os pesos aplicados sobre `V`.

#### Intuição de Q, K e V

Uma analogia útil para entender os três vetores:

- **Query (Q)** — *"o que este token está procurando?"*: representa a consulta feita por um token ao resto da sequência.
- **Key (K)** — *"que informação este token oferece para ser encontrado?"*: funciona como um índice ou rótulo de cada token candidato.
- **Value (V)** — *"qual conteúdo será efetivamente combinado na saída?"*: o dado que é misturado proporcionalmente ao peso de atenção.

O modelo compara a consulta (`Q`) com os índices (`K`) para decidir quanto cada token importa. Depois, usa esses pesos para combinar os conteúdos (`V`). Se dois tokens são fortemente relacionados no contexto, o produto entre seus vetores `Q` e `K` será maior, aumentando o peso de atenção entre eles.

#### Cálculo passo a passo

A fórmula completa:

$$
\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

pode ser decomposta em quatro etapas:

**1. Escores brutos** — produto escalar entre cada par Query↔Key:

$$S = QK^T \quad (S_{ij} \text{ mede a afinidade entre o token } i \text{ e o token } j)$$

**2. Escala** — divide por $\sqrt{d_k}$ para estabilidade numérica:

$$\tilde{S} = \frac{S}{\sqrt{d_k}}$$

> **Por que $\sqrt{d_k}$?** Sem a divisão, os valores de $QK^T$ crescem proporcionalmente a $d_k$. Com dimensões altas, o softmax fica muito "pontudo" (quase one-hot), saturando gradientes e dificultando o treinamento. A escala mantém os logits em uma faixa controlada.

**3. Normalização** — softmax linha a linha transforma escores em pesos probabilísticos (somam 1 por token):

$$A = \mathrm{softmax}(\tilde{S}), \quad A_{ij} \in [0, 1], \quad \sum_j A_{ij} = 1$$

**4. Combinação ponderada dos valores:**

$$H = AV \quad \text{(saída contextualizada)}$$

#### Exemplo concreto

Para uma sequência de 3 tokens, os pesos de atenção calculados para o token 1 poderiam ser:

$$A_{1,\cdot} = [0{,}10,\ 0{,}75,\ 0{,}15]$$

A representação de saída do token 1 é então:

$$h_1 = 0{,}10\,v_1 + 0{,}75\,v_2 + 0{,}15\,v_3$$

O token 1 "presta 75% de atenção" ao token 2 — sua representação final carrega informação contextual dos vizinhos mais relevantes, não apenas de si mesmo.

### 4. Construção do Método Softmax

A função softmax é uma função de ativação amplamente utilizada em redes neurais, especialmente em cenários de classificação, nos quais é importante transformar valores brutos de saída (*logits*) em probabilidades que somam 1.

No contexto de atenção, ela converte os escores em pesos normalizados, destacando os elementos mais relevantes.

A formulação matemática da softmax para cada elemento $i$ é:

$$
\mathrm{softmax}(x_i) = \frac{e^{x_i}}{\sum_{j=1}^{n} e^{x_j}}
$$

Abaixo, está a implementação da função softmax com comentários em cada etapa do cálculo.

In [4]:
# função softmax

def softmax(x):

    # calcula a exponencial de cada elemento do input ajustado pelo maximo valor do input
    # para evitar overflow numerico
    e = np.exp(x-np.max(x))

    # divide cada exponencial pelo somatorio dos exponenciais ao longo do ultimo aixo (axis = 1)
    # o reshape(-1,1) garante que a divisao seja realizada corretamente em um contexto multidimensional
    return e/e.sum(axis=-1).reshape(-1,1)

### 5. Scale Dot Product

A função `scaled_dot_product_attention()` é um componente central do mecanismo de atenção em modelos Transformer. Ela calcula a atenção entre conjuntos de *queries* (`Q`), *keys* (`K`) e *values* (`V`).

Em termos práticos, essa função calcula escores com produto escalar entre `Q` e `K`, aplica a escala por $\sqrt{d_k}$, utiliza `softmax` para obter pesos normalizados e combina esses pesos com `V`.

Esse processo permite que o modelo atribua diferentes níveis de importância às partes da entrada, tornando os Transformers especialmente eficazes em tarefas de PLN e outras tarefas sequenciais.

In [5]:
# escrevendo a função scale dot product
def scale_dot_product_attention(Q, K, V):

    # calcula o produto escalar entre Q e a transposta de K
    matmul_qk = np.dot(Q, K.T)

    # obtem a dimensão dos vetores de chave
    depth = K.shape[-1]

    # escala os logits dividindo-os pela raiz quadrada da profundidade
    logits = matmul_qk/np.sqrt(depth)

    # aplica a função softmax para obter os pesos de atenção
    attention_weights = softmax(logits)

    # multiplica os pesos de atenção pelos valores V para obter a saida final
    
    return np.dot(attention_weights, V)

### 6. Operador Linear combinado com Softmax

In [6]:
# A função linear_softmax() combina uma camada linear seguida de softmax,
# estrutura comum em modelos de aprendizado profundo para classificação.
def linear_softmax(input):

    # Inicializa uma matriz de pesos aleatórios.
    # Essa matriz conecta a dimensão do modelo (dim_model)
    # ao tamanho do vocabulário (vocab_size).
    weights = np.random.randn(dim_model, vocab_size)

    # Aplica a transformação linear (produto entre entrada e pesos).
    # O resultado, chamado logits, representa escores brutos para cada classe/token.
    logits = np.dot(input, weights)

    # Aplica softmax para converter logits em probabilidades,
    # de forma que a soma dos valores no último eixo seja 1.
    return softmax(logits)

### 7. Modelo Final

In [7]:
def model(input):

    # 1) Converte os índices dos tokens de entrada em vetores densos (embeddings).
    # Cada token passa a ser representado em um espaço vetorial contínuo.
    embedded_input = embeddings(input, vocab_size, dim_model)

    # 2) Aplica self-attention (Q = K = V = embedded_input).
    # O objetivo é combinar informações de todos os tokens para cada posição da sequência.
    attention_output = scale_dot_product_attention(embedded_input, embedded_input, embedded_input)

    # 3) Projeta a saída da atenção para o espaço do vocabulário
    # e transforma os logits em probabilidades com softmax.
    output_prob = linear_softmax(attention_output)

    # 4) Seleciona, para cada posição, o token de maior probabilidade.
    # axis=-1 indica que a escolha é feita na dimensão do vocabulário.
    output_idx = np.argmax(output_prob, axis=-1)

    # 5) Retorna os índices previstos.
    return output_idx

### 8. Realizando predições com o Modelo

In [8]:
# gerando dados aleatorios
input_seq = np.random.randint(0, vocab_size, seq_length)
input_seq

array([98, 60,  0, 31,  8, 86, 22, 26, 12, 24])

In [9]:
len(input_seq)

10

In [10]:
output = model(input_seq)
output

array([40, 75, 62, 20, 23, 18, 90, 90, 45, 94])

---

<div align="center">

## <span style="color:#1E90FF;">Treinando um Modelo</span>

</div>

Para esse caso, utilizaremos um modelo de embeddings robusto.

In [11]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import numpy as np

load_dotenv(override = True)
openai_api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=openai_api_key)

def embeddings_openai(input_text, model="text-embedding-3-small"):
    # aceita uma string única ou uma lista de strings
    is_single_input = isinstance(input_text, str)
    texts = [input_text] if is_single_input else list(input_text)

    # chama a API de embeddings
    response = client.embeddings.create(
        model=model,
        input=texts
    )

    # converte a resposta em array numpy
    vectors = np.array([item.embedding for item in response.data], dtype=np.float32)

    # retorna vetor 1D para string única e matriz 2D para lista
    return vectors[0] if is_single_input else vectors

In [12]:
res = embeddings_openai("Oi Julles como vai?")
res

array([ 0.00190248, -0.0137709 , -0.03702377, ..., -0.03176116,
       -0.00312395,  0.03171479], dtype=float32)

### 9. Dataset de sentimentos e pré-processamento

Agora vamos usar o `sentimentos_data.csv` para treinar um classificador com arquitetura inspirada no Transformer.

**Importante:**
- toda a arquitetura será implementada com `NumPy`;
- os embeddings serão obtidos com a função `embeddings_openai()` já definida acima.

In [13]:
import csv
import re
from pathlib import Path

# caminho do dataset
csv_path = Path("dataset/sentimentos_data.csv")

texts = []
labels_text = []
with csv_path.open("r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        texts.append(row["Input"].strip())
        labels_text.append(row["Output"].strip())

classes = sorted(set(labels_text))
label_to_id = {label: idx for idx, label in enumerate(classes)}
id_to_label = {idx: label for label, idx in label_to_id.items()}
y_all = np.array([label_to_id[label] for label in labels_text], dtype=np.int64)

print(f"Total de amostras: {len(texts)}")
print("Classes:", classes)
print("Mapeamento:", label_to_id)


# tokenização simples (mantém letras com acentos usando regex unicode)
def tokenize(text):
    return re.findall(r"\w+", text.lower(), flags=re.UNICODE)


# cache de embeddings para evitar chamadas repetidas para os mesmos tokens
embedding_cache = {}

def get_token_embeddings(tokens):
    unique_tokens = [tok for tok in sorted(set(tokens)) if tok not in embedding_cache]
    if unique_tokens:
        vecs = embeddings_openai(unique_tokens)  # usa função definida no topo do notebook
        if vecs.ndim == 1:
            vecs = vecs[None, :]
        for tok, vec in zip(unique_tokens, vecs):
            embedding_cache[tok] = vec.astype(np.float32)

    return np.array([embedding_cache[tok] for tok in tokens], dtype=np.float32)


max_len = 8
X_seq = []
M_seq = []

for txt in texts:
    toks = tokenize(txt)
    if not toks:
        toks = ["vazio"]

    toks = toks[:max_len]
    tok_vecs = get_token_embeddings(toks)  # (n_tokens, emb_dim)
    emb_dim = tok_vecs.shape[1]

    x = np.zeros((max_len, emb_dim), dtype=np.float32)
    m = np.zeros((max_len, 1), dtype=np.float32)

    x[:len(toks)] = tok_vecs
    m[:len(toks)] = 1.0

    X_seq.append(x)
    M_seq.append(m)

X_all = np.stack(X_seq, axis=0)  # (N, T, D_in)
M_all = np.stack(M_seq, axis=0)  # (N, T, 1)

print("Shape X_all:", X_all.shape)
print("Shape M_all:", M_all.shape)
print("Shape y_all:", y_all.shape)

Total de amostras: 180
Classes: ['felicidade', 'nulo', 'tristeza']
Mapeamento: {'felicidade': 0, 'nulo': 1, 'tristeza': 2}
Shape X_all: (180, 8, 1536)
Shape M_all: (180, 8, 1)
Shape y_all: (180,)


### 10. Transformer Encoder do zero (NumPy)

Implementaremos um bloco `Encoder` inspirado na figura (atenção + residual, feed-forward + residual), com saída para classificação.

Para manter o treino viável sem autograd, usamos uma versão compacta:
- self-attention de 1 cabeça;
- sem `LayerNorm`;
- treinamento com backprop manual em `NumPy`.

> Mesmo simplificado, o fluxo segue o coração do Transformer: projeções `Q/K/V`, atenção escalada, residual e `feed-forward`.

In [14]:
# ---------- utilitários matemáticos ----------
def softmax_axis(x, axis=-1):
    x_shift = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x_shift)
    return ex / (np.sum(ex, axis=axis, keepdims=True) + 1e-12)


def positional_encoding(seq_len, d_model):
    pe = np.zeros((seq_len, d_model), dtype=np.float32)
    pos = np.arange(seq_len)[:, None]
    i = np.arange(d_model)[None, :]
    angles = pos / np.power(10000, (2 * (i // 2)) / d_model)
    pe[:, 0::2] = np.sin(angles[:, 0::2])
    pe[:, 1::2] = np.cos(angles[:, 1::2])
    return pe


# ---------- inicialização de parâmetros ----------
rng = np.random.default_rng(42)

N, T, D_in = X_all.shape
n_classes = len(classes)

d_model = 32
d_ff = 64

params = {
    # projeção de entrada para d_model
    "W_in": 0.02 * rng.standard_normal((D_in, d_model), dtype=np.float32),
    "b_in": np.zeros(d_model, dtype=np.float32),

    # projeções de atenção
    "W_q": 0.02 * rng.standard_normal((d_model, d_model), dtype=np.float32),
    "b_q": np.zeros(d_model, dtype=np.float32),
    "W_k": 0.02 * rng.standard_normal((d_model, d_model), dtype=np.float32),
    "b_k": np.zeros(d_model, dtype=np.float32),
    "W_v": 0.02 * rng.standard_normal((d_model, d_model), dtype=np.float32),
    "b_v": np.zeros(d_model, dtype=np.float32),
    "W_o": 0.02 * rng.standard_normal((d_model, d_model), dtype=np.float32),
    "b_o": np.zeros(d_model, dtype=np.float32),

    # feed-forward
    "W1": 0.02 * rng.standard_normal((d_model, d_ff), dtype=np.float32),
    "b1": np.zeros(d_ff, dtype=np.float32),
    "W2": 0.02 * rng.standard_normal((d_ff, d_model), dtype=np.float32),
    "b2": np.zeros(d_model, dtype=np.float32),

    # cabeça de classificação
    "W_cls": 0.02 * rng.standard_normal((d_model, n_classes), dtype=np.float32),
    "b_cls": np.zeros(n_classes, dtype=np.float32),
}

PE = positional_encoding(T, d_model)


def forward_only(x, mask, p):
    # x: (T, D_in), mask: (T, 1)
    X = (x @ p["W_in"] + p["b_in"] + PE) * mask

    Q = X @ p["W_q"] + p["b_q"]
    K = X @ p["W_k"] + p["b_k"]
    V = X @ p["W_v"] + p["b_v"]

    scores = (Q @ K.T) / np.sqrt(d_model)
    scores = scores + (1.0 - mask.T) * (-1e9)  # mascara de chaves (padding)

    A = softmax_axis(scores, axis=-1)
    A = A * mask  # zera consultas de padding

    H = A @ V
    O = H @ p["W_o"] + p["b_o"]

    R1 = X + O
    Z1 = R1 @ p["W1"] + p["b1"]
    G = np.maximum(Z1, 0.0)
    F = G @ p["W2"] + p["b2"]
    R2 = R1 + F

    valid = float(np.clip(mask.sum(), 1.0, 1e9))
    pooled = (R2 * mask).sum(axis=0) / valid

    logits = pooled @ p["W_cls"] + p["b_cls"]
    probs = softmax_axis(logits[None, :], axis=-1)[0]
    return probs


def forward_backward(x, mask, y_true, p):
    grads = {k: np.zeros_like(v) for k, v in p.items()}

    # ----- forward -----
    X_pre = x @ p["W_in"] + p["b_in"] + PE
    X = X_pre * mask

    Q = X @ p["W_q"] + p["b_q"]
    K = X @ p["W_k"] + p["b_k"]
    V = X @ p["W_v"] + p["b_v"]

    scores = (Q @ K.T) / np.sqrt(d_model)
    scores = scores + (1.0 - mask.T) * (-1e9)

    A = softmax_axis(scores, axis=-1)
    A = A * mask

    H = A @ V
    O = H @ p["W_o"] + p["b_o"]

    R1 = X + O
    Z1 = R1 @ p["W1"] + p["b1"]
    G = np.maximum(Z1, 0.0)
    F = G @ p["W2"] + p["b2"]
    R2 = R1 + F

    valid = float(np.clip(mask.sum(), 1.0, 1e9))
    pooled = (R2 * mask).sum(axis=0) / valid

    logits = pooled @ p["W_cls"] + p["b_cls"]
    probs = softmax_axis(logits[None, :], axis=-1)[0]

    loss = -np.log(probs[y_true] + 1e-12)

    # ----- backward: classificação -----
    dlogits = probs.copy()
    dlogits[y_true] -= 1.0

    grads["W_cls"] = np.outer(pooled, dlogits)
    grads["b_cls"] = dlogits

    dpooled = dlogits @ p["W_cls"].T
    dR2 = (mask / valid) * dpooled  # broadcast para (T, d_model)

    # residual 2
    dR1 = dR2.copy()
    dF = dR2

    # FFN
    grads["W2"] = G.T @ dF
    grads["b2"] = dF.sum(axis=0)

    dG = dF @ p["W2"].T
    dZ1 = dG * (Z1 > 0.0)

    grads["W1"] = R1.T @ dZ1
    grads["b1"] = dZ1.sum(axis=0)

    dR1 += dZ1 @ p["W1"].T

    # residual 1
    dX = dR1.copy()
    dO = dR1

    # O = H @ W_o + b_o
    grads["W_o"] = H.T @ dO
    grads["b_o"] = dO.sum(axis=0)
    dH = dO @ p["W_o"].T

    # H = A @ V
    dA = dH @ V.T
    dV = A.T @ dH

    # A = softmax(scores)
    dScores = A * (dA - (dA * A).sum(axis=-1, keepdims=True))

    scale = np.sqrt(d_model)
    dQ = (dScores @ K) / scale
    dK = (dScores.T @ Q) / scale

    # Q, K, V projeções
    grads["W_q"] = X.T @ dQ
    grads["b_q"] = dQ.sum(axis=0)
    grads["W_k"] = X.T @ dK
    grads["b_k"] = dK.sum(axis=0)
    grads["W_v"] = X.T @ dV
    grads["b_v"] = dV.sum(axis=0)

    dX += dQ @ p["W_q"].T
    dX += dK @ p["W_k"].T
    dX += dV @ p["W_v"].T

    # X = X_pre * mask
    dX_pre = dX * mask

    # entrada
    grads["W_in"] = x.T @ dX_pre
    grads["b_in"] = dX_pre.sum(axis=0)

    return loss, grads, probs


def evaluate(X, M, y, p):
    preds = []
    losses = []
    for i in range(len(X)):
        probs = forward_only(X[i], M[i], p)
        pred = int(np.argmax(probs))
        loss = -np.log(probs[y[i]] + 1e-12)
        preds.append(pred)
        losses.append(loss)
    preds = np.array(preds, dtype=np.int64)
    acc = float((preds == y).mean())
    return float(np.mean(losses)), acc


# split treino/teste
indices = np.arange(N)
rng.shuffle(indices)
cut = int(0.8 * N)
idx_train, idx_test = indices[:cut], indices[cut:]

X_train, M_train, y_train = X_all[idx_train], M_all[idx_train], y_all[idx_train]
X_test, M_test, y_test = X_all[idx_test], M_all[idx_test], y_all[idx_test]

print(f"Treino: {len(X_train)} | Teste: {len(X_test)}")

Treino: 144 | Teste: 36


In [15]:
# ---------- treino ----------
learning_rate = 5e-3
epochs = 2000

for epoch in range(1, epochs + 1):
    order = np.arange(len(X_train))
    rng.shuffle(order)

    epoch_loss = 0.0
    for j in order:
        loss, grads, _ = forward_backward(X_train[j], M_train[j], y_train[j], params)
        epoch_loss += loss

        # SGD
        for k in params:
            params[k] -= learning_rate * grads[k]

    if epoch % 200 == 0 or epoch == 1:
        train_loss, train_acc = evaluate(X_train, M_train, y_train, params)
        test_loss, test_acc = evaluate(X_test, M_test, y_test, params)
        print(
            f"Epoch {epoch:03d} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.3f} | "
            f"test_loss={test_loss:.4f} test_acc={test_acc:.3f}"
        )

Epoch 001 | train_loss=1.0985 train_acc=0.312 | test_loss=1.0944 test_acc=0.417


/var/folders/yp/cbz_mzxs6bz3xnpkml_fqr_h0000gn/T/ipykernel_31549/2394054457.py:91: RuntimeWarning: divide by zero encountered in matmul
  X_pre = x @ p["W_in"] + p["b_in"] + PE
/var/folders/yp/cbz_mzxs6bz3xnpkml_fqr_h0000gn/T/ipykernel_31549/2394054457.py:91: RuntimeWarning: overflow encountered in matmul
  X_pre = x @ p["W_in"] + p["b_in"] + PE
/var/folders/yp/cbz_mzxs6bz3xnpkml_fqr_h0000gn/T/ipykernel_31549/2394054457.py:91: RuntimeWarning: invalid value encountered in matmul
  X_pre = x @ p["W_in"] + p["b_in"] + PE
/var/folders/yp/cbz_mzxs6bz3xnpkml_fqr_h0000gn/T/ipykernel_31549/2394054457.py:105: RuntimeWarning: divide by zero encountered in matmul
  O = H @ p["W_o"] + p["b_o"]
/var/folders/yp/cbz_mzxs6bz3xnpkml_fqr_h0000gn/T/ipykernel_31549/2394054457.py:105: RuntimeWarning: overflow encountered in matmul
  O = H @ p["W_o"] + p["b_o"]
/var/folders/yp/cbz_mzxs6bz3xnpkml_fqr_h0000gn/T/ipykernel_31549/2394054457.py:105: RuntimeWarning: invalid value encountered in matmul
  O = H @ p[

Epoch 200 | train_loss=0.0014 train_acc=1.000 | test_loss=0.2387 test_acc=0.861
Epoch 400 | train_loss=0.0003 train_acc=1.000 | test_loss=0.2420 test_acc=0.861
Epoch 600 | train_loss=0.0001 train_acc=1.000 | test_loss=0.2467 test_acc=0.861
Epoch 800 | train_loss=0.0001 train_acc=1.000 | test_loss=0.2504 test_acc=0.861
Epoch 1000 | train_loss=0.0001 train_acc=1.000 | test_loss=0.2532 test_acc=0.861
Epoch 1200 | train_loss=0.0000 train_acc=1.000 | test_loss=0.2556 test_acc=0.861
Epoch 1400 | train_loss=0.0000 train_acc=1.000 | test_loss=0.2574 test_acc=0.861
Epoch 1600 | train_loss=0.0000 train_acc=1.000 | test_loss=0.2590 test_acc=0.861
Epoch 1800 | train_loss=0.0000 train_acc=1.000 | test_loss=0.2606 test_acc=0.861
Epoch 2000 | train_loss=0.0000 train_acc=1.000 | test_loss=0.2620 test_acc=0.861


In [16]:
# ---------- predições ----------
def predict_text(text, p):
    toks = tokenize(text)
    if not toks:
        toks = ["vazio"]
    toks = toks[:max_len]

    tok_vecs = get_token_embeddings(toks)
    x = np.zeros((max_len, tok_vecs.shape[1]), dtype=np.float32)
    m = np.zeros((max_len, 1), dtype=np.float32)

    x[:len(toks)] = tok_vecs
    m[:len(toks)] = 1.0

    probs = forward_only(x, m, p)
    pred_id = int(np.argmax(probs))
    return id_to_label[pred_id], probs


examples = [
    "Hoje acordei muito feliz e com energia",
    "Estou me sentindo para baixo e sem motivação",
]

for sentence in examples:
    pred_label, prob = predict_text(sentence, params)
    print(f"Texto: {sentence}")
    print(f"Predição: {pred_label}")
    print(f"Probabilidades: {np.round(prob, 4)}")
    print("-" * 80)

Texto: Hoje acordei muito feliz e com energia
Predição: felicidade
Probabilidades: [1. 0. 0.]
--------------------------------------------------------------------------------


/var/folders/yp/cbz_mzxs6bz3xnpkml_fqr_h0000gn/T/ipykernel_31549/2394054457.py:58: RuntimeWarning: divide by zero encountered in matmul
  X = (x @ p["W_in"] + p["b_in"] + PE) * mask
/var/folders/yp/cbz_mzxs6bz3xnpkml_fqr_h0000gn/T/ipykernel_31549/2394054457.py:58: RuntimeWarning: overflow encountered in matmul
  X = (x @ p["W_in"] + p["b_in"] + PE) * mask
/var/folders/yp/cbz_mzxs6bz3xnpkml_fqr_h0000gn/T/ipykernel_31549/2394054457.py:58: RuntimeWarning: invalid value encountered in matmul
  X = (x @ p["W_in"] + p["b_in"] + PE) * mask
/var/folders/yp/cbz_mzxs6bz3xnpkml_fqr_h0000gn/T/ipykernel_31549/2394054457.py:71: RuntimeWarning: divide by zero encountered in matmul
  O = H @ p["W_o"] + p["b_o"]
/var/folders/yp/cbz_mzxs6bz3xnpkml_fqr_h0000gn/T/ipykernel_31549/2394054457.py:71: RuntimeWarning: overflow encountered in matmul
  O = H @ p["W_o"] + p["b_o"]
/var/folders/yp/cbz_mzxs6bz3xnpkml_fqr_h0000gn/T/ipykernel_31549/2394054457.py:71: RuntimeWarning: invalid value encountered in matmul


Texto: Estou me sentindo para baixo e sem motivação
Predição: tristeza
Probabilidades: [0.0037 0.     0.9963]
--------------------------------------------------------------------------------


### 11. Discussão dos resultados

Após o treinamento, é esperado observar queda de `loss` e aumento de `accuracy`, especialmente no conjunto de treino. Como o dataset é pequeno e com frases relativamente curtas, o modelo tende a aprender padrões lexicais importantes para cada classe:

- palavras como `feliz`, `incrível`, `orgulhoso` tendem a reforçar `felicidade`;
- palavras como `triste`, `abatido`, `difícil` tendem a reforçar `tristeza`;
- frases descritivas e operacionais (ex.: `atualizar`, `reunião`, `arquivo`) tendem a cair em `nulo`.

Mesmo com essa simplicidade, o mecanismo de atenção permite que o modelo considere o contexto da frase inteira, em vez de olhar token por token de forma isolada.

### 12. Explicações conceituais

A implementação construída segue a lógica central de um Transformer Encoder:

1. **Embeddings + posição**
   - Cada token da frase é convertido em vetor denso (`embeddings_openai`).
   - A codificação posicional adiciona noção de ordem dos tokens.

2. **Self-Attention**
   - A partir de `X`, criamos `Q`, `K` e `V` por projeções lineares.
   - Calculamos escores `QK^T / sqrt(d_k)` e aplicamos `softmax` para obter pesos de atenção.
   - Esses pesos combinam `V`, produzindo uma representação contextualizada.

3. **Conexões residuais + Feed-Forward**
   - O bloco residual preserva informação da entrada e melhora estabilidade de otimização.
   - A rede feed-forward adiciona não linearidade e capacidade de representação.

4. **Pooling e classificação**
   - Como o problema é classificação de sentença, agregamos os vetores dos tokens válidos.
   - A saída passa por camada linear e `softmax` para prever a classe.

Essa sequência reproduz o fluxo didático da arquitetura da figura: embeddings, atenção, bloco transformador e saída probabilística.

### 13. Limitações e possíveis melhorias

Embora funcional, esta versão é intencionalmente simplificada para fins pedagógicos:

- usa `single-head attention` (ao invés de multi-head);
- não inclui `LayerNorm`;
- utiliza treino com gradientes manuais e SGD simples;
- depende de um dataset pequeno (risco de overfitting e variância alta em teste).

Melhorias recomendadas para evolução:

- adicionar **multi-head attention**;
- incluir **LayerNorm** no padrão `Add & Norm`;
- empilhar múltiplos blocos encoder (`N x`);
- usar regularização (dropout, weight decay);
- ampliar o dataset e validar com `k-fold`.

Esses passos aproximam o modelo da arquitetura completa dos Transformers modernos.

### 14. Conclusão

Neste notebook, desenvolvemos um classificador inspirado em Transformer **do zero em Python/NumPy**, sem uso de `torch`, conectando teoria e prática em três camadas:

- **representação** (embeddings com OpenAI + informação posicional),
- **modelagem contextual** (self-attention + feed-forward residual),
- **decisão** (camada linear com softmax para classificação).

O resultado mostra que, mesmo em uma implementação compacta, os princípios da arquitetura Transformer já são suficientes para aprender padrões semânticos úteis no problema de sentimentos.

Como próximos passos, a evolução para `multi-head`, `LayerNorm` e empilhamento de blocos torna o modelo mais próximo da arquitetura completa apresentada na figura e prepara o caminho para tarefas mais complexas.